In [2]:
# Cell 1: Import and connect
import psycopg2
import pandas as pd
import yaml

with open('../configs/config.yml', 'r') as f:
    config = yaml.safe_load(f)

db = config['database']
conn = psycopg2.connect(
    host=db['host'],
    port=db['port'],
    database=db['database'],
    user=db['user'],
    password=db['password']
)

print("Connected to database")

Connected to database


In [5]:
# Cell 2: Requests by status
df_status = pd.read_sql("""
    SELECT status, COUNT(*) as count
    FROM requests
    GROUP BY status
    ORDER BY count DESC
""", conn)

print("REQUESTS BY STATUS")
print(df_status.to_string(index=False))

REQUESTS BY STATUS
 status  count
FETCHED      1


/tmp/ipykernel_27243/2534971492.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_status = pd.read_sql("""


In [6]:
# Cell 3: Detailed request list with counts
df_requests = pd.read_sql("""
    SELECT 
        r.req_id,
        r.status,
        r.created_at,
        COUNT(DISTINCT rc.comment_id) as raw_count,
        COUNT(DISTINCT pc.comment_id) as preprocessed_count,
        COUNT(DISTINCT p.comment_id) as predicted_count
    FROM requests r
    LEFT JOIN raw_comments rc ON r.req_id = rc.req_id
    LEFT JOIN preprocessed_comments pc ON r.req_id = pc.req_id
    LEFT JOIN predictions p ON r.req_id = p.req_id
    GROUP BY r.req_id, r.status, r.created_at
    ORDER BY r.req_id DESC
""", conn)

print("DETAILED REQUESTS")
df_requests

DETAILED REQUESTS


/tmp/ipykernel_27243/2795544877.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_requests = pd.read_sql("""


,req_id,status,created_at,raw_count,preprocessed_count,predicted_count
0,1,FETCHED,2026-05-22 22:01:17.406723,9,7,0


In [7]:
# Cell 4: Raw comments by request
df_raw = pd.read_sql("""
    SELECT req_id, COUNT(*) as comment_count
    FROM raw_comments
    GROUP BY req_id
    ORDER BY req_id DESC
""", conn)

print("RAW COMMENTS PER REQUEST")
df_raw

RAW COMMENTS PER REQUEST


/tmp/ipykernel_27243/2342364965.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql("""


,req_id,comment_count
0,1,9


In [8]:
# Cell 5: Preprocessed comments by request
df_pre = pd.read_sql("""
    SELECT req_id, COUNT(*) as preprocessed_count
    FROM preprocessed_comments
    GROUP BY req_id
    ORDER BY req_id DESC
""", conn)

print("PREPROCESSED COMMENTS PER REQUEST")
df_pre

PREPROCESSED COMMENTS PER REQUEST


/tmp/ipykernel_27243/1841494490.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pre = pd.read_sql("""


,req_id,preprocessed_count
0,1,7


In [9]:
# Cell 7: Overall summary
summary = pd.read_sql("""
    SELECT 
        (SELECT COUNT(*) FROM requests) as total_requests,
        (SELECT COUNT(*) FROM requests WHERE status = 'NEW') as new_requests,
        (SELECT COUNT(*) FROM requests WHERE status = 'FETCHED') as fetched_requests,
        (SELECT COUNT(*) FROM requests WHERE status = 'PREPROCESSING') as preprocessing_requests,
        (SELECT COUNT(*) FROM requests WHERE status = 'FINALIZED') as finalized_requests,
        (SELECT COUNT(*) FROM requests WHERE status = 'FAILED') as failed_requests,
        (SELECT COUNT(*) FROM raw_comments) as total_raw_comments,
        (SELECT COUNT(*) FROM preprocessed_comments) as total_preprocessed,
        (SELECT COUNT(*) FROM predictions) as total_predictions
""", conn)

print("SUMMARY STATISTICS")
summary

SUMMARY STATISTICS


/tmp/ipykernel_27243/1077813733.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  summary = pd.read_sql("""


,total_requests,new_requests,fetched_requests,preprocessing_requests,finalized_requests,failed_requests,total_raw_comments,total_preprocessed,total_predictions
0,1,0,1,0,0,0,9,7,0
